In [1]:
%pip install torch ta mplfinance scikit-learn matplotlib pandas numpy

import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import os
import pandas as pd
import math

# --- הגדרת המטבע ---
SYMBOL = 'BTCUSDT'
MODEL_TYPE = 'transformer'

# --- 1. חיבור לגוגל דרייב ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/CryptoProject'
except ImportError:
    BASE_DIR = os.getcwd()

# Shared module (crypto_eval.py in the CryptoProject folder). Trainer only saves a bundle.
sys.path.append(BASE_DIR)
from crypto_eval import save_eval_bundle

# --- הגדרות נתיבים מעודכנות לפי המטבע ---
DATA_DIR = os.path.join(BASE_DIR, 'processed_data_transformer', SYMBOL)
MODEL_DIR = os.path.join(BASE_DIR, 'models')
if not os.path.exists(MODEL_DIR): os.makedirs(MODEL_DIR)

BATCH_SIZE = 128
EPOCHS = 50
LEARNING_RATE = 0.001
D_MODEL = 16
N_HEADS = 4
NUM_LAYERS = 1
DROPOUT = 0.4
EARLY_STOP_PATIENCE = 5

# ---------------------------------------------------------
# 1. TFT Components: GLU, GRN, VSN
# ---------------------------------------------------------
class GLU(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.fc = nn.Linear(input_size, input_size * 2)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.fc(x)
        content, gate = torch.chunk(x, 2, dim=-1)
        return content * self.sigmoid(gate)

class GRN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size=None, dropout=0.1):
        super().__init__()
        output_size = output_size or input_size
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, output_size)
        self.glu = GLU(output_size)
        self.layer_norm = nn.LayerNorm(output_size)
        self.dropout = nn.Dropout(dropout)
        self.skip = nn.Linear(input_size, output_size) if input_size != output_size else nn.Identity()

    def forward(self, x):
        residual = self.skip(x)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        x = self.dropout(x)
        x = self.glu(x)
        return self.layer_norm(residual + x)

class VariableSelectionNetwork(nn.Module):
    def __init__(self, input_dim, num_vars, d_model, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.num_vars = num_vars
        self.grns = nn.ModuleList([GRN(input_dim // num_vars, d_model, d_model, dropout) for _ in range(num_vars)])
        self.selector_grn = GRN(input_dim, d_model, num_vars, dropout)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):
        weights = self.softmax(self.selector_grn(x))
        var_outputs = []
        chunk_size = x.shape[-1] // self.num_vars
        for i in range(self.num_vars):
            var_x = x[..., i*chunk_size : (i+1)*chunk_size]
            var_outputs.append(self.grns[i](var_x))
        var_outputs = torch.stack(var_outputs, dim=-1)
        selected_output = torch.sum(var_outputs * weights.unsqueeze(-2), dim=-1)
        return selected_output

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class TFTModel(nn.Module):
    def __init__(self, input_dim, num_vars, d_model=64, nhead=4, num_layers=2, dropout=0.1):
        super().__init__()
        self.vsn = VariableSelectionNetwork(input_dim, num_vars, d_model, dropout)
        self.pos_encoder = PositionalEncoding(d_model)
        encoder_layers = nn.TransformerEncoderLayer(d_model, nhead, d_model*4, dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layers, num_layers)
        self.fc = nn.Linear(d_model, 1)

    def forward(self, x):
        x = self.vsn(x)
        x = self.pos_encoder(x)
        x = self.transformer(x)
        # No sigmoid: Bollinger %B is not bounded to [0,1].
        return self.fc(x[:, -1, :])

# ---------------------------------------------------------
# 2. Training Logic
# ---------------------------------------------------------
class EarlyStopping:
    def __init__(self, patience=5, verbose=False, delta=0):
        self.patience = patience
        self.verbose = verbose
        self.delta = delta
        self.best_score = None
        self.early_stop = False
        self.counter = 0
        self.best_loss = np.inf

    def __call__(self, val_loss, model, model_save_path):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model, model_save_path)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f'  EarlyStopping: {self.counter}/{self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model, model_save_path)
            self.counter = 0

    def save_checkpoint(self, val_loss, model, model_save_path):
        torch.save(model.state_dict(), model_save_path)
        self.best_loss = val_loss

def train():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    print(f"Loading data from: {DATA_DIR}")

    X_train = torch.tensor(np.load(os.path.join(DATA_DIR, 'X_train.npy')), dtype=torch.float32).to(device)
    y_train = torch.tensor(np.load(os.path.join(DATA_DIR, 'y_train.npy')), dtype=torch.float32).to(device)
    X_val = torch.tensor(np.load(os.path.join(DATA_DIR, 'X_val.npy')), dtype=torch.float32).to(device)
    y_val = torch.tensor(np.load(os.path.join(DATA_DIR, 'y_val.npy')), dtype=torch.float32).to(device)
    X_test = torch.tensor(np.load(os.path.join(DATA_DIR, 'X_test.npy')), dtype=torch.float32).to(device)
    y_test = np.load(os.path.join(DATA_DIR, 'y_test.npy'))

    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)

    model = TFTModel(input_dim=X_train.shape[2], num_vars=X_train.shape[2], d_model=D_MODEL).to(device)

    criterion = nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-2)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

    model_save_path = os.path.join(MODEL_DIR, f'best_tft_vsn_{SYMBOL}.pth')
    early_stopping = EarlyStopping(patience=EARLY_STOP_PATIENCE, verbose=True, delta=1e-6)

    train_losses, val_losses = [], []

    for epoch in range(EPOCHS):
        model.train()
        total_train_loss = 0
        for bx, by in train_loader:
            optimizer.zero_grad()
            pred = model(bx).squeeze()
            loss = criterion(pred, by)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()

        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for bx, by in val_loader:
                pred = model(bx).squeeze()
                total_val_loss += criterion(pred, by).item()

        avg_val = total_val_loss / len(val_loader)
        scheduler.step(avg_val)
        train_losses.append(total_train_loss/len(train_loader))
        val_losses.append(avg_val)
        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {total_train_loss/len(train_loader):.6f} | Val Loss: {avg_val:.6f}")

        early_stopping(avg_val, model, model_save_path)
        if early_stopping.early_stop:
            print(f"Early stopping at epoch {epoch+1}")
            break

    # --- Predict on test set, then SAVE the results bundle (graphs are made in the eval notebook) ---
    model.load_state_dict(torch.load(model_save_path))
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X_test), BATCH_SIZE):
            preds.extend(model(X_test[i:i+BATCH_SIZE]).squeeze().cpu().numpy())
    preds = np.array(preds).flatten()

    save_eval_bundle(BASE_DIR, SYMBOL, MODEL_TYPE, y_test, preds, train_losses, val_losses)
    print(f"Done. Now run the eval notebook:  generate('{SYMBOL}', '{MODEL_TYPE}')")

if __name__ == "__main__":
    train()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda
Loading data from: /content/drive/MyDrive/CryptoProject/processed_data_transformer/BTCUSDT
Epoch 1/50 | Train Loss: 0.091207 | Val Loss: 0.034968
Epoch 2/50 | Train Loss: 0.037411 | Val Loss: 0.032737
Epoch 3/50 | Train Loss: 0.034987 | Val Loss: 0.031923
Epoch 4/50 | Train Loss: 0.033970 | Val Loss: 0.031119
Epoch 5/50 | Train Loss: 0.033093 | Val Loss: 0.031046
Epoch 6/50 | Train Loss: 0.032684 | Val Loss: 0.030222
Epoch 7/50 | Train Loss: 0.032299 | Val Loss: 0.030539
  EarlyStopping: 1/5
Epoch 8/50 | Train Loss: 0.031910 | Val Loss: 0.029781
Epoch 9/50 | Train Loss: 0.031752 | Val Loss: 0.029684
Epoch 10/50 | Train Loss: 0.031323 | Val Loss: 0.029840
  EarlyStopping: 1/5
Epoch 11/50 | Train Loss: 0.031120 | Val Loss: 0.029721
  EarlyStopping: 2/5
Epoch 12/50 | Train Loss: 0.031096 | Val Loss: 0.029547
Epoch 13/50 | Train Loss: 0.030862 